In [1]:
######## snakemake preamble start (automatically inserted, do not edit) ########
import sys;sys.path.extend(['/home/wamesjang/anaconda3/envs/snakemake/lib/python3.13/site-packages', '/home/wamesjang/rockefeller/diagram_dynamic_metabolism_v2/workflow', '/home/wamesjang/anaconda3/envs/snakemake/bin', '/home/wamesjang/anaconda3/envs/snakemake/lib/python3.13', '/home/wamesjang/anaconda3/envs/snakemake/lib/python3.13/lib-dynload', '/home/wamesjang/anaconda3/envs/snakemake/lib/python3.13/site-packages', '/home/wamesjang/.cache/snakemake/snakemake/source-cache/runtime-cache/tmp5wuq37ch/file/home/wamesjang/rockefeller/diagram_dynamic_metabolism_v2/workflow/notebooks', '/home/wamesjang/rockefeller/diagram_dynamic_metabolism_v2/workflow/notebooks']);import pickle;from snakemake import script;script.snakemake = pickle.loads(b'\x80\x04\x95\xa8\x03\x00\x00\x00\x00\x00\x00\x8c\x10snakemake.script\x94\x8c\tSnakemake\x94\x93\x94)\x81\x94}\x94(\x8c\x05input\x94\x8c\x0csnakemake.io\x94\x8c\nInputFiles\x94\x93\x94)\x81\x94}\x94(\x8c\x06_names\x94}\x94\x8c\x12_allowed_overrides\x94]\x94(\x8c\x05index\x94\x8c\x04sort\x94eh\x0fh\x06\x8c\x0eAttributeGuard\x94\x93\x94)\x81\x94}\x94\x8c\x04name\x94h\x0fsbh\x10h\x12)\x81\x94}\x94h\x15h\x10sbub\x8c\x06output\x94h\x06\x8c\x0bOutputFiles\x94\x93\x94)\x81\x94\x8c"resources/data/six_state_trees.pkl\x94a}\x94(h\x0b}\x94h\r]\x94(h\x0fh\x10eh\x0fh\x12)\x81\x94}\x94h\x15h\x0fsbh\x10h\x12)\x81\x94}\x94h\x15h\x10sbub\x8c\r_params_store\x94h\x06\x8c\x06Params\x94\x93\x94)\x81\x94}\x94(h\x0b}\x94h\r]\x94(h\x0fh\x10eh\x0fh\x12)\x81\x94}\x94h\x15h\x0fsbh\x10h\x12)\x81\x94}\x94h\x15h\x10sbub\x8c\r_params_types\x94}\x94\x8c\twildcards\x94h\x06\x8c\tWildcards\x94\x93\x94)\x81\x94}\x94(h\x0b}\x94h\r]\x94(h\x0fh\x10eh\x0fh\x12)\x81\x94}\x94h\x15h\x0fsbh\x10h\x12)\x81\x94}\x94h\x15h\x10sbub\x8c\x07threads\x94K\x01\x8c\tresources\x94h\x06\x8c\tResources\x94\x93\x94)\x81\x94(K\x01K\x01\x8c\x04/tmp\x94e}\x94(h\x0b}\x94(\x8c\x06_cores\x94K\x00N\x86\x94\x8c\x06_nodes\x94K\x01N\x86\x94\x8c\x06tmpdir\x94K\x02N\x86\x94uh\r]\x94(h\x0fh\x10eh\x0fh\x12)\x81\x94}\x94h\x15h\x0fsbh\x10h\x12)\x81\x94}\x94h\x15h\x10sbhDK\x01hFK\x01hHhAub\x8c\x03log\x94h\x06\x8c\x03Log\x94\x93\x94)\x81\x94\x8c&logs/run_fig2_partition_function.ipynb\x94a}\x94(h\x0b}\x94\x8c\x08notebook\x94K\x00N\x86\x94sh\r]\x94(h\x0fh\x10eh\x0fh\x12)\x81\x94}\x94h\x15h\x0fsbh\x10h\x12)\x81\x94}\x94h\x15h\x10sbhVhSub\x8c\x06config\x94}\x94\x8c\x04rule\x94\x8c\x18fig_2_partition_function\x94\x8c\x0fbench_iteration\x94N\x8c\tscriptdir\x94\x8cL/home/wamesjang/rockefeller/diagram_dynamic_metabolism_v2/workflow/notebooks\x94ub.');del script;from snakemake.logging import logger;from snakemake.script import snakemake;import os; os.chdir(r'/home/wamesjang/rockefeller/diagram_dynamic_metabolism_v2');
######## snakemake preamble end #########


In [2]:
# start coding here
import numpy as np
from itertools import product
from multiprocessing import Pool
import pickle
import mpmath
import networkx as nx

import numpy as np
from itertools import product
from multiprocessing import Pool
import pickle
import mpmath
import networkx as nx

def spanTrees(trs, Edg, all_span_trees, k):
    if k == 0:
        all_span_trees.extend(product(*trs))#Expand parallels edges using the Cartesian product
    
    for i in range(k):
        if Edg[k][i] == []: continue
        trs.append(Edg[k][i])
        Edg[i] = [Edg[i][j] + Edg[k][j] for j in range(i)]#Contraction 
        spanTrees(trs, Edg, all_span_trees, k-1)
        trs.pop()
        [Edg[i][j].pop() for j in range(i) for _ in range(len(Edg[k][j]))]#Deletion

#Helper function
def Spanning_Trees_Generator(g, hch = False):
    if not nx.is_connected(g):
        print("Only for connected graphs")
        return 0
    if nx.is_directed(g): nx.to_undirected(g)
    n = g.number_of_nodes()
    edgs = list(g.edges)
    Edg = [[[] for _ in range(n)] for _ in range(n)]
    mx = len(edgs)
    edgNum = dict() #dictionary designed to store labeled edges where the keys are integer labels for the edges, and the values are tuples representing the ordinary edge definitions (in, out)

    for ed in edgs:
        i, j = sorted(ed)
        Edg[j][i] = [mx]
        edgNum[mx] = ed
        mx -= 1
    all_span_trees = []
    spanTrees([], Edg, all_span_trees, n-1)
  
    if hch:  
        return all_span_trees #Generate only the labeled edges (Keys) 
    else:
        return [nx.Graph(edgNum[k] for k in element) for element in all_span_trees] #Generate spanning trees as NetworkX graphics objects


In [3]:
# Calculation of partition function
G = nx.Graph()
G.add_nodes_from([2, 1, 0, 3, 4, 5])
G.add_edges_from([(0, 1), (1, 2), (0,2), (3,4), (4,5), (3,5), (0,3), (1,4), (2,5)])
spanTreesGraph = Spanning_Trees_Generator(G)

spanTreesGraph[14] = np.nan
spanTreesGraph[17] = np.nan
spanTreesGraph[27] = np.nan
spanTreesGraph[30] = np.nan
spanTreesGraph[33] = np.nan
spanTreesGraph[58] = np.nan
spanTreesGraph[61] = np.nan
spanTreesGraph[70] = np.nan
spanTreesGraph[73] = np.nan
spanTreesGraph = [g for g in spanTreesGraph if type(g) != float]



In [4]:
with open(snakemake.output[0], 'wb') as f:
    pickle.dump(spanTreesGraph, f)